# 03 — Model Training

This notebook builds and trains the LSTM model for all four ETFs:
- **Model architecture** — Bidirectional LSTM with attention layer
- **Training** — early stopping, validation monitoring, loss curves
- **Saving** — trained weights and scaler saved to `models/`

Expected runtime: ~5–15 minutes per ETF depending on hardware.

## 1. Imports

In [ ]:
import sys
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
sys.path.append(os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

from data_loader import TICKERS, TRAIN_START, TRAIN_END, download_data, load_macro_data
from features import engineer_features
from model import (
    FEATURES, SEQUENCE_LENGTH, MODEL_DIR,
    prepare_sequences, split_sequences,
    build_model, train_model, save_model
)

print(f'TensorFlow version : {tf.__version__}')
print(f'Sequence length    : {SEQUENCE_LENGTH} days')
print(f'Features           : {len(FEATURES)}')
print(f'Model save dir     : {MODEL_DIR}')
gpu = tf.config.list_physical_devices('GPU')
print(f'GPU available      : {"Yes — " + gpu[0].name if gpu else "No — training on CPU"}')

## 2. Model Architecture Summary

Print the model architecture before training so the layer shapes are visible.

In [ ]:
input_shape = (SEQUENCE_LENGTH, len(FEATURES))
sample_model = build_model(input_shape)
sample_model.summary()
print(f'\nInput shape  : {input_shape}  → (timesteps, features)')
print(f'Output shape : (1,)  → predicted Close price (scaled)')

## 3. Load Processed Data

Load from `data/processed/` if available, otherwise re-engineer from raw.

In [ ]:
featured_data = {}

for ticker in TICKERS:
    csv_path = f"data/processed/{ticker.replace('.', '_')}_featured.csv"
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
        print(f'Loaded {ticker} from processed CSV ({len(df)} rows, {df.shape[1]} features)')
    else:
        print(f'Re-engineering features for {ticker}...')
        raw = download_data(ticker, TRAIN_START, TRAIN_END)
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)
        macro = load_macro_data(TRAIN_START, TRAIN_END)
        df = engineer_features(raw, macro)
    featured_data[ticker] = df

print('\nDone.')

## 4. Build Sequences for All ETFs

In [ ]:
sequence_data = {}

for ticker in TICKERS:
    df = featured_data[ticker]
    X, y, scaler = prepare_sequences(df)
    X_train, X_val, y_train, y_val = split_sequences(X, y)
    sequence_data[ticker] = {
        'X_train': X_train, 'X_val': X_val,
        'y_train': y_train, 'y_val': y_val,
        'scaler': scaler
    }
    print(f'{ticker}:  X_train {X_train.shape}  |  X_val {X_val.shape}')

## 5. Train All ETFs

Trains one model per ETF. Early stopping monitors validation loss with patience=10 — training halts automatically when the model stops improving. Each model and its scaler are saved to `models/` after training.

In [ ]:
training_histories = {}

for ticker in TICKERS:
    print(f"\n{'='*55}")
    print(f'  Training {ticker}')
    print(f"{'='*55}")

    data = sequence_data[ticker]
    model = build_model((data['X_train'].shape[1], data['X_train'].shape[2]))

    history = train_model(
        model,
        data['X_train'], data['y_train'],
        data['X_val'],   data['y_val']
    )

    save_model(model, data['scaler'], ticker)
    training_histories[ticker] = history

    best_epoch = np.argmin(history.history['val_loss']) + 1
    best_val   = min(history.history['val_loss'])
    print(f'  Best epoch : {best_epoch}  |  Best val loss : {best_val:.6f}')

print(f"\n{'='*55}")
print('All models trained and saved.')

## 6. Training Loss Curves

Train vs validation loss per epoch for each ETF. A well-trained model shows both curves converging — validation loss flattening is what triggers early stopping.

In [ ]:
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, ticker in enumerate(TICKERS):
    hist = training_histories[ticker].history
    epochs = range(1, len(hist['loss']) + 1)
    best_epoch = np.argmin(hist['val_loss']) + 1

    axes[i].plot(epochs, hist['loss'],     color=colors[i], linewidth=1.5, label='Train loss')
    axes[i].plot(epochs, hist['val_loss'], color=colors[i], linewidth=1.5, label='Val loss',
                 linestyle='--', alpha=0.7)
    axes[i].axvline(best_epoch, color='black', linestyle=':', linewidth=1,
                    label=f'Best epoch ({best_epoch})')
    axes[i].set_title(f'{ticker} — Loss Curve')
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('MSE Loss')
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Training vs Validation Loss — All ETFs', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Training Summary Table

In [ ]:
rows = []
for ticker in TICKERS:
    hist = training_histories[ticker].history
    best_epoch  = np.argmin(hist['val_loss']) + 1
    total_epochs = len(hist['loss'])
    rows.append({
        'Ticker'      : ticker,
        'Total epochs': total_epochs,
        'Best epoch'  : best_epoch,
        'Final train loss': f"{hist['loss'][-1]:.6f}",
        'Best val loss'   : f"{min(hist['val_loss']):.6f}",
        'Model saved' : f"models/{ticker}_model.keras"
    })

summary = pd.DataFrame(rows).set_index('Ticker')
display(summary)

## 8. Overfitting Check

A large gap between train and validation loss indicates overfitting. Plot the gap across epochs for all four ETFs on a single chart.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

for i, ticker in enumerate(TICKERS):
    hist = training_histories[ticker].history
    gap  = np.array(hist['val_loss']) - np.array(hist['loss'])
    ax.plot(range(1, len(gap) + 1), gap, color=colors[i], linewidth=1.5, label=ticker)

ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.fill_between(range(1, 101), 0, 0.01, alpha=0.05, color='red', label='Overfitting zone')
ax.set_title('Val Loss − Train Loss per Epoch (overfitting gap)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss gap')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(left=1)

plt.tight_layout()
plt.show()

print('A gap near 0 means the model generalises well.')
print('A large positive gap means validation loss >> train loss → overfitting.')

## 9. Verify Saved Models

In [ ]:
from tensorflow.keras.models import load_model as keras_load
from model import AttentionLayer
import joblib

print('Verifying saved models...\n')
for ticker in TICKERS:
    model_path  = f'{MODEL_DIR}/{ticker}_model.keras'
    scaler_path = f'{MODEL_DIR}/{ticker}_scaler.pkl'

    model_ok  = os.path.exists(model_path)
    scaler_ok = os.path.exists(scaler_path)

    if model_ok and scaler_ok:
        loaded = keras_load(model_path, custom_objects={'AttentionLayer': AttentionLayer})
        scaler = joblib.load(scaler_path)
        print(f'{ticker}:  model ✓  scaler ✓  (input shape: {loaded.input_shape})')
    else:
        print(f'{ticker}:  model {"✓" if model_ok else "✗"}  scaler {"✓" if scaler_ok else "✗"}')

print('\nAll models verified and ready for evaluation.')

## Summary

| | |
|---|---|
| Architecture | Bidirectional LSTM (128) → LSTM (64) → Attention → Dense (32) → Dense (1) |
| Optimiser | Adam |
| Loss function | Mean Squared Error |
| Early stopping | Patience = 10, restores best weights |
| Dropout | 20% after each LSTM layer |
| Models saved | `models/{ticker}_model.keras` |
| Scalers saved | `models/{ticker}_scaler.pkl` |

**Next → `04_evaluation.ipynb`** — load saved models, run predictions on 2025 data, and measure accuracy.